# Router or one agent

**Scenario:** one agent holds five alert playbooks: phishing, beaconing, exfiltration, password
spray, and close as noise. It demos well. In the queue it quietly runs nothing for many alerts.

Nothing errored. The model named the right playbook in plain English, then asked a question.

The fix is **a hospital triage desk**. One person decides which department you belong to. That
department has one job and the tools for it.

## Mechanics

A tool call is the model asking your code to run a named function, as data, not as an action. A
branch holding one tool can ask for that one thing and nothing else.

| Piece | What it is | What it decides |
|---|---|---|
| `StateGraph(State)` | The graph, typed by a `TypedDict` | Which keys nodes may write |
| `add_edge(START, node)` | The entry edge | Where a run begins |
| `add_conditional_edges(node, fn, names)` | A branch | Which node runs next |
| `fn(state) -> str` | The routing function | It returns a node name, nothing else |
| `tools=[...]` per branch | What is in scope there | What the model is able to ask for |
| `tool_choice` | Names the one tool to answer through | It forces the call instead of asking for it |

The routing function returns a node name. A name with no node raises `KeyError` mid run, so an
unknown label crashes rather than misroutes.

## The picture

![One agent with five tools against a classify step and five branches](images/router-or-one-agent.svg)

The top path holds everything at once. The bottom decides first, then acts with one tool in scope.

## The cost

The cost is not the bill. It is the alerts that reach nobody.

```
missed = alerts x share of turns that return no tool call
```

An alert that runs no playbook looks like a quiet night.

## The failure

Six alerts, five playbooks, one agent. Start with the queue.

In [1]:
from vault import get_client, load_env, model_for

load_env()
client = get_client("02-multi-agent-orchestration/01-router-or-one-agent")

ALERTS = [
    ("phishing", "Mail gateway quarantined 340 messages spoofing the CFO display name, three users replied before quarantine."),
    ("beaconing", "Host WKS-4471 makes a 512 byte DNS TXT query to a newly registered domain every 61 seconds for nine hours."),
    ("exfil", "Service account svc-backup copied 42 GB from the finance share to an external S3 bucket at 03:10."),
    ("bruteforce", "Password spray against Entra ID, one password tried against 1900 accounts over four hours."),
    ("phishing", "User clicked a link in a mail from no-reply@0ffice365-secure.com and entered credentials on the landing page."),
    ("noise", "Backup agent restarted twice overnight during the scheduled maintenance window, no other signal."),
]

Then the playbooks, each a tool the model can ask for, and one prompt describing all five.

In [2]:
PLAYBOOKS = {
    "phishing": "Contain credential phishing: reset the user, pull the mail, block the sender domain.",
    "beaconing": "Investigate command and control beaconing: pull netflow, sinkhole the domain, isolate the host.",
    "exfil": "Investigate data exfiltration: pull the transfer records, size it, notify legal.",
    "bruteforce": "Respond to password spray: lock accounts, force a second factor, block the source ranges.",
    "noise": "Close the alert as expected activity and record the suppression reason.",
}

TOOLS = [{"type": "function", "function": {
    "name": f"run_{name}_playbook", "description": what,
    "parameters": {"type": "object", "properties": {"alert_id": {"type": "string"}},
                   "required": ["alert_id"], "additionalProperties": False}}}
    for name, what in PLAYBOOKS.items()]

SYSTEM = ("You are a security operations analyst. Pick exactly one playbook for the alert. "
          + " ".join(f"run_{n}_playbook: {w}" for n, w in PLAYBOOKS.items()))

One turn per alert. It returns what the model asked to run, and what it said when it asked for
nothing.

In [3]:
def pick_playbook(alert):
    """One agent, all five playbooks in scope."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=200, tools=TOOLS,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": alert}])
    message = reply.choices[0].message
    calls = message.tool_calls or []
    return (calls[0].function.name if calls else None), (message.content or "")

One pass is an anecdote, because the same alert is not treated the same twice. Run the queue three
times and count what reached a playbook.

In [4]:
rounds, excuses = [], []
for _ in range(3):
    picks = [pick_playbook(text) for _, text in ALERTS]
    rounds.append([name for name, _ in picks])
    excuses += [said for name, said in picks if name is None]
    print(f"  {sum(1 for name, _ in picks if name)}/{len(ALERTS)} alerts reached a playbook")

missed = sum(1 for run in rounds for name in run if name is None)
print(f"\ndropped {missed} of {len(rounds) * len(ALERTS)} alerts")
if excuses:
    print(f"one of them replied: {excuses[0][:88]!r}")
assert missed == 0, f"{missed} alerts were named in prose and then never ran"

  2/6 alerts reached a playbook


  6/6 alerts reached a playbook


  4/6 alerts reached a playbook

dropped 6 of 18 alerts
one of them replied: 'I can help with this. What is the alert ID?'


AssertionError: 6 alerts were named in prose and then never ran

## The diagnosis

Read the reply. The model identified the alert correctly, then asked for the alert id.
Classification was never the problem.

**`tool_calls` was empty and the run ended normally.** The field on the response that says why the
model stopped reads `stop`, not an error. A dropped alert and a handled one look identical to code
written as `tool_calls or []`.

**Everything was in scope at once.** Five tools and a paragraph about each turned one decision into a
conversation, so the model asked a question back.

**Deciding and acting shared a turn.** Being unsure about one field stalled the whole action.

## The fix

Split the turn. One call classifies and holds no tools. The branch it picks holds one tool, so the
only thing it can ask for is the playbook already chosen. Then take the choice away entirely, because
`tool_choice` forces the call once the decision is upstream.

In [5]:
LABELS = list(PLAYBOOKS)


def classify(alert):
    """One question, no tools in scope. Returns a label or refuses."""
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=10,
        messages=[{"role": "system", "content":
                   f"Reply with one word from this list and nothing else: {', '.join(LABELS)}."},
                  {"role": "user", "content": alert}])
    word = (reply.choices[0].message.content or "").strip().lower().strip(".")
    if word not in LABELS:
        raise ValueError(f"router returned {word!r}, which has no branch")
    return word

The branch node next. Same model, same alert, one tool instead of five. The flag lets the branch
pick freely or be forced, so the difference is measured rather than assumed.

In [6]:
def run_branch(label, state):
    """One branch, one tool. The choice was already made upstream."""
    name = f"run_{label}_playbook"
    forced = {"type": "function", "function": {"name": name}} if state["forced"] else "auto"
    reply = client.chat.completions.create(
        model=model_for("default"), max_tokens=200, tool_choice=forced,
        tools=[t for t in TOOLS if t["function"]["name"] == name],
        messages=[{"role": "system", "content": "Run the playbook for this alert."},
                  {"role": "user", "content": state["alert"]}])
    calls = reply.choices[0].message.tool_calls or []
    return {"ran": [calls[0].function.name] if calls else []}

Now the graph. A run enters at `classify` and the routing function returns the branch name. Every
branch is the same node bound to its own label.

In [7]:
from functools import partial
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class Triage(TypedDict):
    alert: str
    label: str
    ran: list
    forced: bool


graph = StateGraph(Triage)
graph.add_node("classify", lambda s: {"label": classify(s["alert"])})
for name in LABELS:
    graph.add_node(name, partial(run_branch, name))
    graph.add_edge(name, END)
graph.add_edge(START, "classify")
graph.add_conditional_edges("classify", lambda s: s["label"], LABELS)
router = graph.compile()

One helper, so both settings are measured the same way.

In [8]:
def measure(forced):
    """Run every alert through the router and count what executed."""
    outs = [router.invoke({"alert": text, "label": "", "ran": [], "forced": forced})
            for _, text in ALERTS]
    return outs, sum(1 for out in outs if out["ran"])

Same six alerts, same model, same playbooks. Only the shape changed.

In [9]:
loose, loose_ran = measure(False)
strict, strict_ran = measure(True)
correct = sum(1 for (truth, _), out in zip(ALERTS, strict) if out["label"] == truth)
before = sum(1 for run in rounds for name in run if name) / (len(rounds) * len(ALERTS))

print(f"one agent, five tools : ran a playbook for {before:.0%} of alerts")
print(f"router, branch chooses: {loose_ran}/{len(ALERTS)} executed")
print(f"router, call forced   : {strict_ran}/{len(ALERTS)} executed")
print(f"labels correct        : {correct}/{len(ALERTS)}")

one agent, five tools : ran a playbook for 67% of alerts
router, branch chooses: 5/6 executed
router, call forced   : 6/6 executed
labels correct        : 6/6


## The gate

A name with no node is the failure this shape introduces. The check compares the two sets and needs
no model.

In [10]:
def test_every_label_has_a_branch():
    """A label with no node raises KeyError mid run. Catch it at build time."""
    orphans = set(LABELS) - set(graph.nodes)
    assert not orphans, f"the router can return {orphans} and no branch handles it"


test_every_label_has_a_branch()
print(f"gate holds: {len(LABELS)} labels, {len(LABELS)} branches, no orphans")

gate holds: 5 labels, 5 branches, no orphans


Add a label to `PLAYBOOKS` without adding a branch and this test fails.

### Enterprise exploration

- The router doubles the calls per alert. At queue volume, what does that cost per day?
- A misrouted alert runs the wrong playbook confidently. What is the blast radius, and which branch
  carries the worst one?
- An alert matching two playbooks gets one label. Who owns the concern that was dropped?
- The old shape dropped alerts silently. What would have told you, and how late?

### Key takeaways

- An empty `tool_calls` list with a normal stop is a dropped action, not an error.
- Many tools in scope turns one decision into a conversation.
- Classify with no tools, then act with one. Narrowing scope is not enough on its own.
- Once the router has decided, `tool_choice` removes the branch's chance to hesitate.
- A routing function returns a node name, so its labels and your nodes are one set.